# Извлечение + Очистка

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, from_json, to_timestamp
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, FloatType, BooleanType

# 1. Инициализация Spark с Kafka-коннектором
spark = SparkSession.builder \
    .appName("DE_Extract_Clean") \
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.12:3.4.1") \
    .getOrCreate()

# 2. Контракт данных
schema = StructType([
    StructField("event_id", StringType(), True),
    StructField("user_id", IntegerType(), True),
    StructField("event_type", StringType(), True),
    StructField("timestamp", StringType(), True),
    StructField("product_id", IntegerType(), True),
    StructField("amount", FloatType(), True),
    StructField("device", StringType(), True),
    StructField("session_id", StringType(), True),
    StructField("is_test", BooleanType(), True)
])

# 3. Чтение из Kafka + Парсинг JSON
df_raw = spark.read.format("kafka") \
    .option("kafka.bootstrap.servers", "kafka:29092") \
    .option("subscribe", "raw_events") \
    .option("startingOffsets", "earliest") \
    .load()

# 4. Очистка (7 правил)
df_clean = (df_raw.select(from_json(col("value").cast("string"), schema).alias("d"))
    .select("d.*")
    .filter(col("user_id").isNotNull())
    .filter(col("event_type").isNotNull())
    .filter(col("is_test") == False)
    .filter((col("amount").isNull()) | (col("amount") >= 0))
    .withColumn("timestamp", to_timestamp(col("timestamp")))
)

print(f"✅ Исходных: {df_raw.count()} | Очищено: {df_clean.count()}")
df_clean.show(3, truncate=False)

# Трансформация + Сохранение витрины

In [ ]:
from pyspark.sql.functions import to_date, countDistinct, sum as _sum, count
from pyspark.sql.window import Window

# 1. Оконные функции (сессии, паузы)
win = Window.partitionBy("user_id").orderBy("timestamp")
df_enriched = df_clean.withColumn("time_since_last", 
    (col("timestamp").cast("long") - col("timestamp").cast("long").lag().over(win))
)

# 2. Агрегация: 1 пользователь × 1 день
daily_features = (df_enriched.withColumn("event_date", to_date("timestamp"))
    .groupBy("user_id", "event_date")
    .agg(
        countDistinct("session_id").alias("sessions_count"),
        count("*").alias("events_count"),
        _sum("amount").alias("daily_spend"),
        countDistinct("event_type").alias("unique_event_types"),
        count("time_since_last").alias("active_transitions")
    )
)

# 3. Сохранение в Data Lake (Parquet)
daily_features.write.mode("overwrite").parquet("/home/jovyan/work/data/features/")
print("✅ Витрина сохранена: /data/features/")

# 4. Проверка
daily_features.orderBy(col("daily_spend").desc()).show(5, truncate=False)